In [1]:
%pip uninstall statsforecast -y
%pip install statsforecast==1.6.0


Found existing installation: statsforecast 1.6.0
Uninstalling statsforecast-1.6.0:
  Successfully uninstalled statsforecast-1.6.0
Note: you may need to restart the kernel to use updated packages.
  Using cached statsforecast-1.6.0-py3-none-any.whl.metadata (26 kB)
Using cached statsforecast-1.6.0-py3-none-any.whl (110 kB)
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
from pathlib import Path
# Point to project root, not parent of notebooks
sys.path.insert(0, str(Path('..').resolve()))
print(sys.path[0])

C:\Users\rksan_amz5yv3\ev-coldstart-forecast


In [3]:
import sys
from pathlib import Path

# Add src directories directly
sys.path.insert(0, str(Path('..').resolve()))
sys.path.insert(0, str(Path('../src').resolve()))

import warnings
warnings.filterwarnings("ignore")

# Import directly without src prefix
sys.path.insert(0, str(Path('../src/models').resolve()))
sys.path.insert(0, str(Path('../src/evaluation').resolve()))

from baseline import SeasonalNaive, LightGBMBaseline
from metrics import evaluate, temporal_split
print("imports OK")


imports OK


In [4]:
import glob
import pandas as pd
import numpy as np
from pathlib import Path

PROCESSED_DIR = Path('../data/processed')
TIME_COL = "timestamp"
TARGET_COL = "sessions"
HELD_OUT_SITE = "office001"

def load_stations():
    files = glob.glob(str(PROCESSED_DIR / "*.parquet"))
    stations = {}
    for f in files:
        station_id = Path(f).stem
        if HELD_OUT_SITE in station_id.lower():
            continue
        df = pd.read_parquet(f)
        df[TIME_COL] = pd.to_datetime(df[TIME_COL]).dt.tz_localize(None)
        df = df.sort_values(TIME_COL).reset_index(drop=True)
        stations[station_id] = df
    return stations

stations = load_stations()
print(f"Loaded {len(stations)} stations")

Loaded 107 stations


In [5]:
def has_enough_history(df, min_weeks=26):
    span = df[TIME_COL].max() - df[TIME_COL].min()
    return span.days >= (min_weeks * 7)

qualified = {sid: df for sid, df in stations.items() if has_enough_history(df)}
print(f"{len(qualified)} stations qualify with 6+ months of history")

106 stations qualify with 6+ months of history


In [6]:
import mlflow
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("phase2_baselines")

TRAIN_WEEKS_LIST = [1, 2, 3]
EVAL_WEEKS = 4

models = {
    "seasonal_naive": SeasonalNaive,
    "lgbm_vanilla": LightGBMBaseline,
}

all_results = []
total = len(qualified) * len(TRAIN_WEEKS_LIST) * len(models)
count = 0

for station_id, df in qualified.items():
    for train_weeks in TRAIN_WEEKS_LIST:
        train_df, eval_df = temporal_split(df, train_weeks, EVAL_WEEKS, TIME_COL)
        if len(train_df) < 24 or len(eval_df) < 24:
            continue
        y_true = eval_df[TARGET_COL].values

        for model_name, ModelClass in models.items():
            count += 1
            if count % 50 == 0:
                print(f"[{count}/{total}] {station_id} | {train_weeks}w | {model_name}")
            try:
                model = ModelClass()
                model.fit(train_df)
                y_pred = np.clip(model.predict(eval_df), 0, None)
                min_len = min(len(y_true), len(y_pred))
                metrics = evaluate(y_true[:min_len], y_pred[:min_len])

                with mlflow.start_run(run_name=f"{model_name}_{station_id}_{train_weeks}w"):
                    mlflow.set_tags({"station_id": station_id, "model": model_name, "phase": "phase2"})
                    mlflow.log_params({"train_weeks": train_weeks, "eval_weeks": EVAL_WEEKS})
                    mlflow.log_metrics(metrics)

                all_results.append({
                    "station_id": station_id,
                    "model": model_name,
                    "train_weeks": train_weeks,
                    **metrics
                })
            except Exception as e:
                print(f"  FAILED {model_name} {station_id}: {e}")
                continue

print(f"\nDone. {len(all_results)} runs completed.")

[50/636] caltech_2-39-127-19 | 1w | lgbm_vanilla
[100/636] caltech_2-39-131-30 | 2w | lgbm_vanilla
[150/636] caltech_2-39-78-362 | 3w | lgbm_vanilla
[200/636] caltech_2-39-79-379 | 1w | lgbm_vanilla
[250/636] caltech_2-39-83-387 | 2w | lgbm_vanilla
[300/636] caltech_2-39-91-441 | 3w | lgbm_vanilla
[350/636] jpl_1-1-179-777 | 1w | lgbm_vanilla
[400/636] jpl_1-1-179-794 | 2w | lgbm_vanilla
[450/636] jpl_1-1-179-810 | 3w | lgbm_vanilla
[500/636] jpl_1-1-191-789 | 1w | lgbm_vanilla
[550/636] jpl_1-1-191-806 | 2w | lgbm_vanilla
[600/636] jpl_1-1-193-825 | 3w | lgbm_vanilla

Done. 636 runs completed.


In [7]:
results_df = pd.DataFrame(all_results)
print(f"Total results: {len(results_df)}")
print(results_df["model"].value_counts())
print()
summary = results_df.groupby(["model", "train_weeks"])[["mae", "rmse", "mape"]].mean().round(4)
print(summary)

Total results: 636
model
seasonal_naive    318
lgbm_vanilla      318
Name: count, dtype: int64

                               mae    rmse          mape
model          train_weeks                              
lgbm_vanilla   1            0.1024  0.2914  4.868261e+08
               2            0.0899  0.2777  3.165652e+08
               3            0.0864  0.2679  3.248113e+08
seasonal_naive 1            0.3488  0.4655  1.619974e+09
               2            0.3438  0.4577  1.492831e+09
               3            0.3481  0.4559  1.592848e+09
